# MASLD-LLM · C3 evaluations (1.7B-v5 on MB-85xx + MB-87xx)

Colab = compute, Google Drive = storage. Nothing touches the laptop.

This notebook mirrors the platform's validated local-runner protocol
(greedy decoding, left padding, continuation-only decode, first line = answer,
adaptive batch halving) and writes scorer-compatible archives
(`task/sample/answer/raw/error/model_version/parsed`).

Outputs land in Drive at `masld-llm/results/frontier_runs/`:
- `qwen3-1.7b-masld-v5__MB-85xx.jsonl.gz`
- `qwen3-1.7b-masld-v5__MB-87xx.jsonl.gz`

In [ ]:
# Cell 1: Mount Drive + locate the masld-llm workspace
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_MASLD = '/content/drive/MyDrive/masld-llm'
assert os.path.isdir(DRIVE_MASLD), f'not found: {DRIVE_MASLD} — check the Drive account'
PROMPTS = os.path.join(DRIVE_MASLD, 'prompts')
OUT_DIR = os.path.join(DRIVE_MASLD, 'results', 'frontier_runs')
os.makedirs(OUT_DIR, exist_ok=True)
for p in (PROMPTS, OUT_DIR):
    print(p, '->', 'OK' if os.path.isdir(p) else 'MISSING')

In [ ]:
# Cell 2: Dependencies + GPU check (Colab already ships torch/transformers)
!pip install -q -U transformers peft accelerate

import torch, gzip, json, time
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 3: Load corpora from Drive (built 2026-10-18, MANIFEST v3-freeze-10)
import gzip, json

def load_corpus(path):
    recs = []
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('{"__meta__"'):
                continue
            recs.append(json.loads(line))
    return recs

CORPORA = {
    'MB-85xx': 'MB-85xx_progression_test.jsonl.gz',
    'MB-87xx': 'MB-87xx_liverplasma_test.jsonl.gz',
}
corpus_records = {}
for tag, fname in CORPORA.items():
    path = os.path.join(PROMPTS, fname)
    corpus_records[tag] = load_corpus(path)
    tasks = sorted({r['task'] for r in corpus_records[tag]})
    print(f'{tag}: {len(corpus_records[tag])} records | tasks {tasks}')

In [ ]:
# Cell 4: Load Qwen3-1.7B base + the v5 LoRA adapter FROM DRIVE (unmerged = less RAM)
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = 'Qwen/Qwen3-1.7B'
ADAPTER = os.path.join(DRIVE_MASLD, 'checkpoints', 'qwen3-1.7b-masld-sft-v5')
assert os.path.isdir(ADAPTER), f'adapter not found: {ADAPTER}'

tok = AutoTokenizer.from_pretrained(BASE, padding_side='left')
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16)
model = PeftModel.from_pretrained(model, ADAPTER)
model.to('cuda').eval()
print('model ready | adapter:', os.path.basename(ADAPTER))

In [ ]:
# Cell 5: Eval function — mirrors platform local_runner.py protocol
# greedy, max_new_tokens=24, continuation-only decode, first line = answer,
# OOM -> halve batch down to 1. parsed=None (scorer re-derives from raw).
MAX_NEW_TOKENS = 24
MAX_LENGTH = 4096
MODEL_VERSION = 'qwen3-1.7b-masld-v5'

def evaluate(records, batch=8):
    out, i, t0 = [], 0, time.time()
    while i < len(records):
        chunk = records[i:i + batch]
        try:
            enc = tok(
                [r['prompt'] for r in chunk],
                padding=True, truncation=True, max_length=MAX_LENGTH,
                return_tensors='pt',
            ).to('cuda')
            with torch.no_grad():
                gen = model.generate(
                    **enc,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=tok.pad_token_id,
                )
            for j, r in enumerate(chunk):
                text = tok.decode(gen[j][enc['input_ids'].shape[1]:], skip_special_tokens=True)
                first_line = text.split('\n')[0].strip()
                out.append({
                    'task': r['task'],
                    'sample': r['sample'],
                    'answer': r.get('answer'),
                    'raw': first_line,
                    'error': None,
                    'model_version': MODEL_VERSION,
                    'parsed': None,
                })
            i += len(chunk)
            if (i // batch) % 10 == 0 or i == len(records):
                rate = i / max(time.time() - t0, 1e-9)
                print(f'  {i}/{len(records)} ({rate:.1f} items/s)', flush=True)
        except RuntimeError as exc:
            if batch > 1:
                batch = max(batch // 2, 1)
                print(f'  batch failure -> halving to {batch} ({exc})', flush=True)
                continue
            raise
    return out

def write_archive(records, out_path):
    with gzip.open(out_path, 'wt', encoding='utf-8') as f:
        f.write(json.dumps({'__meta__': {
            'n_records': len(records), 'model_version': MODEL_VERSION,
            'protocol': 'greedy, max_new_tokens=24, newline-stopped (platform local_runner)',
        }}) + '\n')
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print('wrote', out_path)

print('eval harness ready')

In [ ]:
# Cell 6: MB-85xx (progression: 8501 stable/progressive, 8505 ordinal)
recs = evaluate(corpus_records['MB-85xx'], batch=8)
write_archive(recs, os.path.join(OUT_DIR, 'qwen3-1.7b-masld-v5__MB-85xx.jsonl.gz'))

In [ ]:
# Cell 7: MB-87xx (liver->plasma: 8701 platform 3-class, 8705 protein level 0-10)
recs = evaluate(corpus_records['MB-87xx'], batch=8)
write_archive(recs, os.path.join(OUT_DIR, 'qwen3-1.7b-masld-v5__MB-87xx.jsonl.gz'))

In [ ]:
# Cell 8: Verify archives + quick accuracy peek (raw vs gold, exact-match)
from collections import Counter

for tag in CORPORA:
    path = os.path.join(OUT_DIR, f'qwen3-1.7b-masld-v5__{tag}.jsonl.gz')
    recs = load_corpus(path)
    print(f'\n=== {tag}: {len(recs)} records ===')
    for task in sorted({r["task"] for r in recs}):
        sub = [r for r in recs if r['task'] == task]
        match = sum(1 for r in sub if str(r['raw']).strip().lower() == str(r['answer']).strip().lower())
        maj = Counter(str(r['answer']) for r in sub).most_common(1)[0]
        print(f'{task}: n={len(sub)} | exact-match {match/len(sub):.3f} '
              f'| majority("{maj[0]}")={maj[1]/len(sub):.3f}')
        print('  raw sample:', Counter(r['raw'] for r in sub).most_common(3))

In [ ]:
# Cell 9 (optional): also evaluate the 0.6B-v5 adapter for the same corpora
# Swap the adapter in-place without reloading the whole session.
RUN_06B = False  # flip to True to run

if RUN_06B:
    from transformers import AutoModelForCausalLM as _AM
    import gc
    del model
    gc.collect(); torch.cuda.empty_cache()
    BASE06 = 'Qwen/Qwen3-0.6B'
    ADAPTER06 = os.path.join(DRIVE_MASLD, 'checkpoints', 'qwen3-0.6b-masld-sft-v5')
    model = _AM.from_pretrained(BASE06, dtype=torch.float16)
    model = PeftModel.from_pretrained(model, ADAPTER06)
    model.to('cuda').eval()
    MODEL_VERSION = 'qwen3-0.6b-masld-v5'
    globals()['MODEL_VERSION'] = MODEL_VERSION
    for tag in CORPORA:
        recs = evaluate(corpus_records[tag], batch=8)
        write_archive(recs, os.path.join(OUT_DIR, f'qwen3-0.6b-masld-v5__{tag}.jsonl.gz'))
else:
    print('skipped (RUN_06B = False)')